# Final Metatable Formation


## 1- Importing all working libraries

In [ ]:

# data handling and application development
import numpy as np
from pandas.api.types import is_numeric_dtype
import pandas as pd
from pathlib import Path
import joblib
import ast


# audio feature extraction (eGeMAPS)
import opensmile

# for data visualisation
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

# for Machine learning preprocessing and validation
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV, LeaveOneGroupOut

# for machine learning models
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from catboost import CatBoostClassifier

# for class imbalance handling
from imblearn.over_sampling import SMOTE

# for model evaluation
from sklearn.metrics import accuracy_score, recall_score, classification_report, confusion_matrix

# for deep embeddings
import torch
import librosa
from transformers import Wav2Vec2Processor, Wav2Vec2Model

# for dimensionality reduction plot
from sklearn.manifold import TSNE

In [ ]:
# reading the data from the stored csv files containing all the datasets metadata infomration
upx_metadata_clean = pd.read_csv("upx_metadata_clean.csv")
uxtd_metadata_clean = pd.read_csv("uxtd_metadata_clean.csv")
uxssd_metadata_clean = pd.read_csv("uxssd_metadata_clean.csv")

In [ ]:
# checking the first 5 rows for upx dataset clean
upx_metadata_clean.head()

In [ ]:
# checking the first 5 rows for uxtd dataset clean
uxtd_metadata_clean.head()

In [ ]:
# checking the first 5 rows for uxssd dataset clean
uxssd_metadata_clean.head()

In [ ]:
# SSD subtype is not available in UXTD and UXSSD dataset, so dropping it
upx_metadata_clean=upx_metadata_clean.drop(columns=["SSD SUBTYPE"])

In [ ]:
# checking upx again
upx_metadata_clean.head()

## 2- Formation of Master Metadata

In [ ]:
master_metadata = pd.concat([upx_metadata_clean,uxtd_metadata_clean,uxssd_metadata_clean], ignore_index=True)

In [ ]:
master_metadata

In [ ]:
# checking data types
master_metadata.dtypes

### Need to form a global speaker ID to avoid recording/data leakage during model training and testing, since similar speaker ID existing in all three datasets.

In [ ]:
master_metadata["global_speaker_id"] = (
    master_metadata["dataset_name"].astype(str) + "_" +
    master_metadata["speaker_id"].astype(str)
)
master_metadata

### Checking for Empty Values

In [ ]:
master_metadata.isna().sum()

In [ ]:
# checking for summary statistics 
master_metadata.describe()

In [ ]:
master_metadata["GENDER"].unique()

#### Gender values need consistency

In [ ]:
# standardizing the gender categories
master_metadata["GENDER"]=master_metadata["GENDER"].replace({"Female":"F","Male":"M"})

In [ ]:
master_metadata["GENDER"].unique()

In [ ]:
# storing the master metadata as a seperate csv file
master_metadata.to_csv("master_metadata.csv", index=False)

In [ ]:
master_metadata.describe(include="all")

In [ ]:
# to make sure we are only having baseline recording sessions in our dataset
master_metadata["session_id"].value_counts()

In [ ]:
master_metadata["GENDER"].value_counts()

In [ ]:
master_metadata["status"].unique()

In [ ]:
# only keeping the processed audio records, since that is the basic assumption for our working
master_metadata = master_metadata[master_metadata["status"] == "processed"]

In [ ]:
master_metadata.shape

In [ ]:
# saving again to csv file (final clean file for master metadata)
master_metadata.to_csv("master_metadata.csv", index=False)

In [ ]:
# checking for prompt text in the dataset and their count
prompt_counts = master_metadata["prompt_text"].value_counts(dropna=False).reset_index()
prompt_counts.columns = ["prompt_text", "count"]
prompt_counts.head(50)

#### Now the prompt text does not contain any irregular information.

## 3- Extraction of Audio Features - eGeMAPS

In [ ]:
# making a copy of the master_metadata but with useful columns only
df=master_metadata[["speaker_id","full_id","processed_child_wav_path","dataset_name","class_label","global_speaker_id"]]
df.shape

### The following code extract eGeMAPS features and store it into a csv file. The code has been comment out for the demo. If testing is required, press command + / (on Mac) to remove all the comments and then run it.

In [ ]:
# smile = opensmile.Smile(
#     feature_set=opensmile.FeatureSet.eGeMAPSv02,
#     feature_level=opensmile.FeatureLevel.Functionals,
# )

# rows = []
# for i, r in df.iterrows(): # because iterrows return two values (index and row)
#     wav = Path(r["processed_child_wav_path"])

#     feat = smile.process_file(str(wav))
#     feat = feat.reset_index(drop=True) # for resetting the index
#     feat.insert(0, "full_id", r["full_id"])
#     feat.insert(1, "global_speaker_id",r["global_speaker_id"])
#     feat.insert(len(feat.columns), "class_label", r["class_label"])
#     rows.append(feat)

# egemaps_df = pd.concat(rows, ignore_index=True)
# print(egemaps_df.shape)
# egemaps_df.head()
# egemaps_df.to_csv("egemaps_features.csv",index=False)

### Since the eGeMAPS features were already extracted and stored in a CSV file (egemaps_features), therefore using the csv file to use those features. In case if someone wants to test the extracted features from the above code, then simply uncomment the above code and comment out the following code line, so that it does not read from the csv file.

In [ ]:
egemaps_df=pd.read_csv("egemaps_features.csv")

In [ ]:
egemaps_df

In [ ]:
# checking for null values
egemaps_df.isna().sum()

In [ ]:
# checking the features names
egemaps_df.columns

## 4- Exploratory Data Analysis

In [ ]:
# checking the distribution of the features
egemaps_df.hist(figsize=(100,100))
plt.show()

In [ ]:
# data integrity checks
print("Missing values total:", egemaps_df.isna().sum().sum())
print("\nClass distribution:")
print(egemaps_df["class_label"].value_counts(dropna=False))


In [ ]:
# storing numerical and categoical features
cat_columns=["full_id","global_speaker_id","class_label"]
num_columns = egemaps_df.drop(columns=cat_columns)
num_columns


In [ ]:
# pearon correlation heatmap to check for identical features
sns.set_context("poster")
plt.figure(figsize=(300,300))
corr= num_columns.corr(numeric_only=True)
sns.heatmap(corr, annot=True, fmt=".2f", linewidths=0.8, linecolor="black", vmin=-1, vmax=1)
plt.title("Correlation Analysis of Audio Features", fontsize=20, fontweight="bold")
plt.show()


#### As 88 features are quite hard to visualize in a single plot, we will extract their values to determine the highly correlated features.

In [ ]:
# getting only the column names of highly correlated features
corr_matrix= num_columns.corr(numeric_only=True)
corr_pairs = corr_matrix.unstack().sort_values(ascending=False)
high_corr = corr_pairs[(abs(corr_pairs) > 0.95) & (corr_pairs.index.get_level_values(0) != corr_pairs.index.get_level_values(1))]
high_corr

**Remarks: Some of the features are related with the profiling section of our speech, like following:**

- loudness_sma3_amean is related to Average loudness
- loudness_sma3_pctlrange0-2 is related to	Loudness variation
- spectralFlux_sma3_amean is related to	Spectral change
- mfcc3_sma3_amean is related to Spectral balance
- alphaRatioUV_sma3nz_amean or hammarbergIndexUV_sma3nz_amean is related to	Voice quality / frequency balance
- F2amplitudeLogRelF0_sma3nz_amean or F3amplitudeLogRelF0_sma3nz_amean is related to Formant/resonance energy.

We have decided to keep them and will remove the rest of them after train test split stage.



### Box Plot Analysis for Outliers

In [ ]:
# non-feature columns
exclude_cols = ["class_label", "global_speaker_id"]

# Numeric feature columns only
feature_cols = [
    c for c in egemaps_df.columns
    if c not in exclude_cols and is_numeric_dtype(egemaps_df[c])
]

print("Number of numeric feature columns:", len(feature_cols))

# grid layout
n_cols = 4
n_rows = int(np.ceil(len(feature_cols) / n_cols))

plt.figure(figsize=(20, 5 * n_rows))

for i, feature in enumerate(feature_cols, 1):
    plt.subplot(n_rows, n_cols, i)
    sns.boxplot(data=egemaps_df, x="class_label", y=feature)
    plt.title(feature, fontsize=9)
    plt.xlabel("class_label")
    plt.ylabel("")
    plt.xticks(rotation=0)

plt.tight_layout()
plt.show()

#### eGeMPAS features have the presence of outliers, which could influence the results. For this purpose, Windsorisation (Outlier Clipping) will be performed later on after data split.

In [ ]:
# to check for class distribution in dataset based on recordings, not speakers
total_class_count = egemaps_df['class_label'].value_counts().sort_index()
print(total_class_count)
plt.figure(figsize=(6, 6))

plt.pie(
    total_class_count,
    labels=["TD", "SSD"],
    autopct='%1.1f%%',
    startangle=90
)

plt.title("Class Label Distribution - Recording Level")
plt.tight_layout()
plt.show()

#### So there is a presence of large imbalance between the number of recordings of SSD class and TD class. For this purpose, two approaches will be followed:

    1- SMOTE 
    2- Using class penalty parameters inside the models

## 5- Validation Test Split

#### One of the working approach for model evelaution is that the data has been split into validation and test data first. On the validation dataset, initially internal model development and evaluation will be performed and after selecting the best model, it will be tested against the test data, which will act as a generalized data.

In [ ]:
# firstly forming speaker level dataframe first
speaker_meta = (
    master_metadata.groupby("global_speaker_id")
    .agg(
        dataset_name=("dataset_name", "first"),
        speaker_id=("speaker_id", "first"),
        class_label=("class_label", "first"),
        gender=("GENDER", "first"),
        age=("AGE", "first"),
        n_utterances=("full_id", "count")
    )
    .reset_index()
)
speaker_meta

#### The test speaker children will be selected based on the matching gender and nearest age gap, if available.

In [ ]:
# speaker_meta columns:
# global_speaker_id, dataset_name, speaker_id, class_label, gender, age, n_utterances

# separating SSD and TD speakers
ssd = speaker_meta[speaker_meta["class_label"] == 1].copy()
td = speaker_meta[speaker_meta["class_label"] == 0].copy()

# sorting SSD speakers so selection is stable
ssd = ssd.sort_values(["gender", "age"]).reset_index(drop=True)

matched_pairs = []
used_td = set()

# for each SSD child, find closest TD child with same gender
for _, ssd_row in ssd.iterrows():

    candidates = td[
        (td["gender"] == ssd_row["gender"]) &
        (~td["global_speaker_id"].isin(used_td))
    ].copy()

    if len(candidates) == 0:
        continue

    candidates["age_diff"] = abs(candidates["age"] - ssd_row["age"])

    best_td = candidates.sort_values(
        ["age_diff", "n_utterances"],
        ascending=[True, False]
    ).iloc[0]

    matched_pairs.append({
        "SSD_speaker": ssd_row["global_speaker_id"],
        "SSD_gender": ssd_row["gender"],
        "SSD_age": ssd_row["age"],
        "SSD_utterances": ssd_row["n_utterances"],

        "TD_speaker": best_td["global_speaker_id"],
        "TD_gender": best_td["gender"],
        "TD_age": best_td["age"],
        "TD_utterances": best_td["n_utterances"],

        "age_diff": best_td["age_diff"]
    })

    used_td.add(best_td["global_speaker_id"])

# converting to dataframe
matched_df = pd.DataFrame(matched_pairs)

# this will allow to seperate out the test speakers from the dataset even before using the data for any sort of model training. For this project, we have selected first 16 sorted children
final_matched= matched_df.sort_values("age_diff").head(8).reset_index(drop=True)

# creating final test speaker list
final_test_speakers = (
    final_matched["SSD_speaker"].tolist() +
    final_matched["TD_speaker"].tolist()
)

print("Matched SSD-TD pairs:")
display(final_matched)

print("\nFinal test speakers:")
print(final_test_speakers)

### Now forming a seperate held out test data for 16 test speakers. These speakers will be used for final generalization.

In [ ]:
final_test_speakers = final_test_speakers

# final held-out test kids
egemaps_final_test = egemaps_df[
    egemaps_df["global_speaker_id"].isin(final_test_speakers)
].copy()

# remaining kids for training / LOSO / validation
egemaps_dev = egemaps_df[
    ~egemaps_df["global_speaker_id"].isin(final_test_speakers)
].copy()

print("Full data:", egemaps_df.shape)
print("DEV data:", egemaps_dev.shape)
print("Final test data:", egemaps_final_test.shape)

print("\nFinal test speakers:")
print(egemaps_final_test["global_speaker_id"].unique())

In [ ]:
# checking the class distribution at child level in held out dataset (for 16 speakers, it should be equally balanced)
print(egemaps_final_test[["global_speaker_id", "class_label"]]
      .drop_duplicates()
      .value_counts("class_label"))

In [ ]:
# to check the number of recording ranges for test kids
egemaps_final_test.groupby(["global_speaker_id", "class_label"]).size()

## Validation dataset split for Model development

In [ ]:
# forming a dictionary to store all the models results 
loso_results_dict = {}
test_results_dict = {}

In [ ]:
# preparing X,y group
non_feature_cols = [
    "full_id",
    "global_speaker_id",
    "class_label"]

non_feature_cols = [c for c in non_feature_cols if c in egemaps_dev.columns]
feature_cols = [c for c in egemaps_dev.columns if c not in non_feature_cols]

X_dev_df = egemaps_dev[feature_cols].copy()
y_dev = egemaps_dev["class_label"].values
groups_dev = egemaps_dev["global_speaker_id"].values # to make sure that during validation, the split is performed at the speaker level, not the recording level

print("X shape:", X_dev_df.shape)
print("Number of speakers:", egemaps_dev["global_speaker_id"].nunique())
print("Class count:")
print(egemaps_dev[["global_speaker_id", "class_label"]].drop_duplicates()["class_label"].value_counts())

#### In the corelation analysis, some features despite having high corelation, were decided not to be dropped because of their need for the speech profiling section, so storing these features names first here.

In [ ]:
# protected profile features
protected_features = [
    "F2amplitudeLogRelF0_sma3nz_amean",
    "loudness_sma3_amean",
    "loudness_sma3_pctlrange0-2",
    "spectralFlux_sma3_amean",
    "mfcc3_sma3_amean",
    "alphaRatioUV_sma3nz_amean"]

protected_features = [f for f in protected_features if f in X_dev_df.columns]

print("Protected features:")
print(protected_features)

#### This function will ensure to retain the protected features, while dropping the rest of the highly correlated features based on the 95% threshold.

In [ ]:
# corelation filter function
def get_high_corr_cols(X_train, threshold=0.95, protected_features=None):
    
    if protected_features is None:
        protected_features = []

    corr = X_train.corr().abs()

    upper = corr.where(
        np.triu(np.ones(corr.shape), k=1).astype(bool)
    )

    drop_cols = []

    for col in upper.columns:
        if any(upper[col] > threshold):
            if col not in protected_features:
                drop_cols.append(col)

    return drop_cols

## 6- Model Development

### Model 1: Support Vector Machine Basic

**Parameters**:
- kernel: rbf
- class_weights: balanced
- Outlier Clipping: Applied for each fold in a loop
- Cross Validation: Leave One Point Out (Here point refers to one group of child)
- Pearon Correlation: 95% Threshold for each fold in a loop
- Standardization: Applied
- Model training is done on the subset of the complete dataset.

In [ ]:
# training on subsample data
logo = LeaveOneGroupOut()
loso_results = []
loso_utterance_preds = []

for fold, (train_idx, test_idx) in enumerate(
    logo.split(X_dev_df, y_dev, groups_dev), start=1):

    X_train = X_dev_df.iloc[train_idx].copy()
    X_test = X_dev_df.iloc[test_idx].copy()

    y_train = y_dev[train_idx]
    y_test = y_dev[test_idx]

    test_speaker = groups_dev[test_idx][0]

    # Outlier clipping for outlier treatment at validation level
    lower = X_train.quantile(0.01)
    upper = X_train.quantile(0.90)

    X_train_clip = X_train.clip(lower=lower, upper=upper, axis=1)
    X_test_clip = X_test.clip(lower=lower, upper=upper, axis=1)

    # Correlation filtering
    corr_drop_cols = get_high_corr_cols(
        X_train_clip,
        threshold=0.95,
        protected_features=protected_features
    )

    X_train_corr = X_train_clip.drop(columns=corr_drop_cols)
    X_test_corr = X_test_clip.drop(columns=corr_drop_cols)

    # Scaling
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_corr)
    X_test_scaled = scaler.transform(X_test_corr)

    # SVM
    svm = SVC(
        kernel="rbf",
        class_weight="balanced",
        random_state=69
    )

    svm.fit(X_train_scaled, y_train)

    y_pred = svm.predict(X_test_scaled)

    # store utterance-level predictions
    temp_utt = pd.DataFrame({
        "fold": fold,
        "test_speaker": test_speaker,
        "global_speaker_id": groups_dev[test_idx],
        "y_true": y_test,
        "y_pred": y_pred
    })

    loso_utterance_preds.append(temp_utt)

    # child-level majority vote
    y_true_child = pd.Series(y_test).mode()[0]
    y_pred_child = pd.Series(y_pred).mode()[0]

    loso_results.append({
        "fold": fold,
        "test_speaker": test_speaker,
        "y_true_child": y_true_child,
        "y_pred_child": y_pred_child,
        "n_utterances": len(y_test),
        "utterance_accuracy": accuracy_score(y_test, y_pred),
        "features_before_corr": X_train_clip.shape[1],
        "features_after_corr": X_train_corr.shape[1],
        "removed_corr_features": len(corr_drop_cols),
        "removed_features": corr_drop_cols
    })

In [ ]:
# utterance level results
loso_utterance_preds = pd.concat(loso_utterance_preds, ignore_index=True)

loso_utt_acc = accuracy_score(
    loso_utterance_preds["y_true"],
    loso_utterance_preds["y_pred"]
)

loso_utt_uar = recall_score(
    loso_utterance_preds["y_true"],
    loso_utterance_preds["y_pred"],
    average="macro",
    zero_division=0
)

loso_utt_recall1 = recall_score(
    loso_utterance_preds["y_true"],
    loso_utterance_preds["y_pred"],
    pos_label=1,
    average="binary",
    zero_division=0
)

print("SVM LOSO Utterance-level Results")

print("\nUtterance-level Confusion Matrix:")
print(confusion_matrix(
    loso_utterance_preds["y_true"],
    loso_utterance_preds["y_pred"]
))

print("\nUtterance-level Classification Report:")
print(classification_report(
    loso_utterance_preds["y_true"],
    loso_utterance_preds["y_pred"],
    zero_division=0
))

print("Utterance-level Accuracy:", loso_utt_acc)
print("Utterance-level UAR:", loso_utt_uar)
print("Utterance-level Class 1 Recall:", loso_utt_recall1)

In [ ]:
# speaker level results at validation data
loso_results = pd.DataFrame(loso_results)

# storing metrics as variables
loso_acc = accuracy_score(
    loso_results["y_true_child"],
    loso_results["y_pred_child"]
)

loso_uar = recall_score(
    loso_results["y_true_child"],
    loso_results["y_pred_child"],
    average="macro",
    zero_division=0
)

loso_recall1 = recall_score(
    loso_results["y_true_child"],
    loso_results["y_pred_child"],
    pos_label=1,
    average="binary",
    zero_division=0
)

# print results
print("SVM with LOSO\n")
print("Child Group Level Results\n")

print("LOSO Child-level Confusion Matrix:")
print(confusion_matrix(
    loso_results["y_true_child"],
    loso_results["y_pred_child"]
))

print("\nLOSO Child-level Classification Report:")
print(classification_report(
    loso_results["y_true_child"],
    loso_results["y_pred_child"],
    zero_division=0
))

print("LOSO Child-level Accuracy:", loso_acc)
print("LOSO Child-level UAR:", loso_uar)
print("LOSO Class 1 Recall:", loso_recall1)

loso_results_dict["Support Vector Machine + eGeMAPS"] = {
    "Accuracy": loso_acc,
    "UAR": loso_uar,
    "Recall_class1": loso_recall1
}

In [ ]:
# now now combining the whole dataset for training on whole
X_train_full = egemaps_dev[feature_cols].copy()
y_train_full = egemaps_dev["class_label"].values

X_test_final = egemaps_final_test[feature_cols].copy()
y_test_final = egemaps_final_test["class_label"].values
groups_test_final = egemaps_final_test["global_speaker_id"].values

print("DEV train shape:", X_train_full.shape)
print("Final test shape:", X_test_final.shape)
print("Final test speakers:", egemaps_final_test["global_speaker_id"].nunique())

lower = X_train_full.quantile(0.01)
upper = X_train_full.quantile(0.90)

X_train_clip = X_train_full.clip(lower=lower, upper=upper, axis=1)
X_test_clip = X_test_final.clip(lower=lower, upper=upper, axis=1)

# Correlation filtering using DEV only
corr_drop_cols = get_high_corr_cols(
    X_train_clip,
    threshold=0.95,
    protected_features=protected_features
)

X_train_corr = X_train_clip.drop(columns=corr_drop_cols)
X_test_corr = X_test_clip.drop(columns=corr_drop_cols)

In [ ]:
# training on whole dataset and prediction on final 16 speakers
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_corr)
X_test_scaled = scaler.transform(X_test_corr)

final_svm = SVC(
    kernel="rbf",
    class_weight="balanced",
    random_state=69
)

final_svm.fit(X_train_scaled, y_train_full)
y_pred_final = final_svm.predict(X_test_scaled)

In [ ]:
# utterance level results on 16 speakers

utt_acc = accuracy_score(y_test_final, y_pred_final)

utt_uar = recall_score(
    y_test_final,
    y_pred_final,
    average="macro",
    zero_division=0
)

utt_recall1 = recall_score(
    y_test_final,
    y_pred_final,
    pos_label=1,
    average="binary",
    zero_division=0
)

print("Final Test Utterance-level Confusion Matrix:")
print(confusion_matrix(y_test_final, y_pred_final))

print("\nFinal Test Utterance-level Classification Report:")
print(classification_report(y_test_final, y_pred_final, zero_division=0))

print("Final Test Utterance-level Accuracy:", utt_acc)
print("Final Test Utterance-level UAR:", utt_uar)



In [ ]:
# results breakdown to show the accuracy bound against the speaker ID
final_pred_df = pd.DataFrame({
    "global_speaker_id": groups_test_final,
    "y_true": y_test_final,
    "y_pred": y_pred_final
})

child_results = []

for speaker_id, group in final_pred_df.groupby("global_speaker_id"):

    child_results.append({
        "global_speaker_id": speaker_id,
        "y_true_child": group["y_true"].mode()[0],
        "y_pred_child": group["y_pred"].mode()[0],
        "n_utterances": len(group),
        "utterance_accuracy": accuracy_score(group["y_true"], group["y_pred"])})

child_results = pd.DataFrame(child_results)

child_results

In [ ]:
# child level results (16 speakers)
test_acc = accuracy_score(
    child_results["y_true_child"],
    child_results["y_pred_child"]
)

test_uar = recall_score(
    child_results["y_true_child"],
    child_results["y_pred_child"],
    average="macro",
    zero_division=0
)

test_recall1 = recall_score(
    child_results["y_true_child"],
    child_results["y_pred_child"],
    pos_label=1,
    average="binary",
    zero_division=0
)

# print results
print("Final Test Child-level Confusion Matrix:")
print(confusion_matrix(
    child_results["y_true_child"],
    child_results["y_pred_child"]
))

print("\nFinal Test Child-level Classification Report:")
print(classification_report(
    child_results["y_true_child"],
    child_results["y_pred_child"],
    zero_division=0
))

print("Final Test Child-level Accuracy:", test_acc)
print("Final Test Child-level UAR:", test_uar)
print("Final Test Class 1 Recall:", test_recall1)


test_results_dict["Support Vector Machine + eGeMAPS"] = {
    "Accuracy": test_acc,
    "UAR": test_uar,
    "Recall_class1": test_recall1,
    "n_kids": child_results.shape[0]
}

### Model 2: Support Vector Machine with SMOTE Oversampling

- SMOTE oversampling is applied at recording level for validation data only during each fold of validation. Rest of the model parameters were kept same as previous

In [ ]:
logo = LeaveOneGroupOut()
loso_results_model2 = []

for fold, (train_idx, test_idx) in enumerate(
    logo.split(X_dev_df, y_dev, groups_dev), start=1
):

    X_train = X_dev_df.iloc[train_idx].copy()
    X_test = X_dev_df.iloc[test_idx].copy()

    y_train = y_dev[train_idx]
    y_test = y_dev[test_idx]

    test_speaker = groups_dev[test_idx][0]

    # Outlier clipping
    lower = X_train.quantile(0.01)
    upper = X_train.quantile(0.90)

    X_train_clip = X_train.clip(lower=lower, upper=upper, axis=1)
    X_test_clip = X_test.clip(lower=lower, upper=upper, axis=1)

    # Correlation filtering
    corr_drop_cols = get_high_corr_cols(
        X_train_clip,
        threshold=0.95,
        protected_features=protected_features
    )

    X_train_corr = X_train_clip.drop(columns=corr_drop_cols)
    X_test_corr = X_test_clip.drop(columns=corr_drop_cols)

    # Scaling
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_corr)
    X_test_scaled = scaler.transform(X_test_corr)

    # SMOTE only on training data
    smote = SMOTE(random_state=69)
    X_train_smote, y_train_smote = smote.fit_resample(
        X_train_scaled,
        y_train
    )

    # SVM model
    svm_smote = SVC(
        kernel="rbf",
        class_weight="balanced",
        random_state=69
    )

    svm_smote.fit(X_train_smote, y_train_smote)

    y_pred = svm_smote.predict(X_test_scaled)

    y_true_child = pd.Series(y_test).mode()[0]
    y_pred_child = pd.Series(y_pred).mode()[0]

    loso_results_model2.append({
        "fold": fold,
        "test_speaker": test_speaker,
        "y_true_child": y_true_child,
        "y_pred_child": y_pred_child,
        "n_utterances": len(y_test),
        "utterance_accuracy": accuracy_score(y_test, y_pred),
        "features_after_corr": X_train_corr.shape[1],
        "class_0_before_smote": np.bincount(y_train)[0],
        "class_1_before_smote": np.bincount(y_train)[1],
        "class_0_after_smote": np.bincount(y_train_smote)[0],
        "class_1_after_smote": np.bincount(y_train_smote)[1]
    })

loso_results_model2 = pd.DataFrame(loso_results_model2)

In [ ]:
# child level results (on validation data)
model2_loso_acc = accuracy_score(
    loso_results_model2["y_true_child"],
    loso_results_model2["y_pred_child"]
)

model2_loso_uar = recall_score(
    loso_results_model2["y_true_child"],
    loso_results_model2["y_pred_child"],
    average="macro",
    zero_division=0
)

model2_loso_recall1 = recall_score(
    loso_results_model2["y_true_child"],
    loso_results_model2["y_pred_child"],
    pos_label=1,
    average="binary",
    zero_division=0
)

print("Model 2: eGeMAPS + SVM + SMOTE")

print("\nLOSO Child-level Confusion Matrix:")
print(confusion_matrix(
    loso_results_model2["y_true_child"],
    loso_results_model2["y_pred_child"]
))

print("\nLOSO Child-level Classification Report:")
print(classification_report(
    loso_results_model2["y_true_child"],
    loso_results_model2["y_pred_child"],
    zero_division=0
))

print("LOSO Child-level Accuracy:", model2_loso_acc)
print("LOSO Child-level UAR:", model2_loso_uar)
print("LOSO Class 1 Recall:", model2_loso_recall1)


loso_results_dict["Support Vector Machine + eGeMAPS + SMOTE"] = {
    "Accuracy": model2_loso_acc,
    "UAR": model2_loso_uar,
    "Recall_class1": model2_loso_recall1
}

In [ ]:
# combining whole dataset again
X_train_full = egemaps_dev[feature_cols].copy()
y_train_full = egemaps_dev["class_label"].values

X_test_final = egemaps_final_test[feature_cols].copy()
y_test_final = egemaps_final_test["class_label"].values
groups_test_final = egemaps_final_test["global_speaker_id"].values


# Outlier clipping
lower = X_train_full.quantile(0.01)
upper = X_train_full.quantile(0.90)

X_train_clip = X_train_full.clip(lower=lower, upper=upper, axis=1)
X_test_clip = X_test_final.clip(lower=lower, upper=upper, axis=1)

# Correlation filtering
corr_drop_cols = get_high_corr_cols(
    X_train_clip,
    threshold=0.95,
    protected_features=protected_features
)

X_train_corr = X_train_clip.drop(columns=corr_drop_cols)
X_test_corr = X_test_clip.drop(columns=corr_drop_cols)

# Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_corr)
X_test_scaled = scaler.transform(X_test_corr)

In [ ]:
# training on complete data
smote = SMOTE(random_state=69)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train_scaled,
    y_train_full
)

print("Before SMOTE:", np.bincount(y_train_full))
print("After SMOTE:", np.bincount(y_train_smote))

final_svm_smote = SVC(
    kernel="rbf",
    class_weight="balanced",
    random_state=69
)

final_svm_smote.fit(X_train_smote, y_train_smote)

y_pred_final_model2 = final_svm_smote.predict(X_test_scaled)

In [ ]:
# Final utterance-level metrics for Model 2 using 16 test speakers

model2_utt_acc = accuracy_score(y_test_final, y_pred_final_model2)

model2_utt_uar = recall_score(
    y_test_final,
    y_pred_final_model2,
    average="macro",
    zero_division=0
)

model2_utt_recall1 = recall_score(
    y_test_final,
    y_pred_final_model2,
    pos_label=1,
    average="binary",
    zero_division=0
)

print("Model 2: Final Test Utterance-level Results")

print("\nUtterance-level Confusion Matrix:")
print(confusion_matrix(y_test_final, y_pred_final_model2))

print("\nUtterance-level Classification Report:")
print(classification_report(
    y_test_final,
    y_pred_final_model2,
    zero_division=0
))

print("Utterance-level Accuracy:", model2_utt_acc)
print("Utterance-level UAR:", model2_utt_uar)
print("Utterance-level Class 1 Recall:", model2_utt_recall1)

In [ ]:
# results breakdown
final_pred_df_model2 = pd.DataFrame({
    "global_speaker_id": groups_test_final,
    "y_true": y_test_final,
    "y_pred": y_pred_final_model2
})

child_results_model2 = []

for speaker_id, group in final_pred_df_model2.groupby("global_speaker_id"):

    child_results_model2.append({
        "global_speaker_id": speaker_id,
        "y_true_child": group["y_true"].mode()[0],
        "y_pred_child": group["y_pred"].mode()[0],
        "n_utterances": len(group),
        "utterance_accuracy": accuracy_score(group["y_true"], group["y_pred"])
    })

child_results_model2 = pd.DataFrame(child_results_model2)

child_results_model2

In [ ]:
# child group level for 16 speakers
model2_test_acc = accuracy_score(
    child_results_model2["y_true_child"],
    child_results_model2["y_pred_child"]
)

model2_test_uar = recall_score(
    child_results_model2["y_true_child"],
    child_results_model2["y_pred_child"],
    average="macro",
    zero_division=0
)

model2_test_recall1 = recall_score(
    child_results_model2["y_true_child"],
    child_results_model2["y_pred_child"],
    pos_label=1,
    average="binary",
    zero_division=0
)

print("Model 2: eGeMAPS + SVM + SMOTE Final Test")

print("\nFinal Test Child-level Confusion Matrix:")
print(confusion_matrix(
    child_results_model2["y_true_child"],
    child_results_model2["y_pred_child"]
))

print("\nFinal Test Child-level Classification Report:")
print(classification_report(
    child_results_model2["y_true_child"],
    child_results_model2["y_pred_child"],
    zero_division=0
))

print("Final Test Child-level Accuracy:", model2_test_acc)
print("Final Test Child-level UAR:", model2_test_uar)
print("Final Test Class 1 Recall:", model2_test_recall1)


test_results_dict["Support Vector Machine + eGeMAPS + SMOTE"] = {
    "Accuracy": model2_test_acc,
    "UAR": model2_test_uar,
    "Recall_class1": model2_test_recall1,
    "n_kids": child_results_model2.shape[0]
}

#### Note: Even applying the SMOTE, the results are only improved by mere 2% on utterance level as compared to the model 1 (SVM with no SMOTE). On child level the results are still same which is 62% (recall for SSD). This shows that class imbalance handling on utterance level did not improve the model performance significantly.

### Model 3: CatBoost
- The ensemble (tree based model) is applied to check whether capturing the non-linear relationship will help improve the classification performance or not.

In [ ]:
print("Model 3: eGeMAPS + CatBoost")

logo = LeaveOneGroupOut()
loso_results_model3 = []

for fold, (train_idx, test_idx) in enumerate(
    logo.split(X_dev_df, y_dev, groups_dev), start=1
):

    X_train = X_dev_df.iloc[train_idx].copy()
    X_test = X_dev_df.iloc[test_idx].copy()

    y_train = y_dev[train_idx]
    y_test = y_dev[test_idx]

    test_speaker = groups_dev[test_idx][0]

    # Outlier clipping
    lower = X_train.quantile(0.01)
    upper = X_train.quantile(0.90)

    X_train_clip = X_train.clip(lower=lower, upper=upper, axis=1)
    X_test_clip = X_test.clip(lower=lower, upper=upper, axis=1)

    # Correlation filtering
    corr_drop_cols = get_high_corr_cols(
        X_train_clip,
        threshold=0.95,
        protected_features=protected_features
    )

    X_train_corr = X_train_clip.drop(columns=corr_drop_cols)
    X_test_corr = X_test_clip.drop(columns=corr_drop_cols)

    # CatBoost does not need scaling
    cat_model = CatBoostClassifier(
        iterations=500,
        learning_rate=0.03,
        depth=6,
        loss_function="Logloss",
        eval_metric="Recall",
        class_weights=[1, 2],
        random_seed=69,
        verbose=0
    )

    cat_model.fit(X_train_corr, y_train)

    y_pred = cat_model.predict(X_test_corr).astype(int).ravel()

    y_true_child = pd.Series(y_test).mode()[0]
    y_pred_child = pd.Series(y_pred).mode()[0]

    loso_results_model3.append({
        "fold": fold,
        "test_speaker": test_speaker,
        "y_true_child": y_true_child,
        "y_pred_child": y_pred_child,
        "n_utterances": len(y_test),
        "utterance_accuracy": accuracy_score(y_test, y_pred),
        "features_after_corr": X_train_corr.shape[1]
    })

loso_results_model3 = pd.DataFrame(loso_results_model3)

In [ ]:
# child level results on LOSO
model3_loso_acc = accuracy_score(
    loso_results_model3["y_true_child"],
    loso_results_model3["y_pred_child"]
)

model3_loso_uar = recall_score(
    loso_results_model3["y_true_child"],
    loso_results_model3["y_pred_child"],
    average="macro",
    zero_division=0
)

model3_loso_recall1 = recall_score(
    loso_results_model3["y_true_child"],
    loso_results_model3["y_pred_child"],
    pos_label=1,
    average="binary",
    zero_division=0
)

print("Model 3: eGeMAPS + CatBoost")

print("\nLOSO Child-level Confusion Matrix:")
print(confusion_matrix(
    loso_results_model3["y_true_child"],
    loso_results_model3["y_pred_child"]
))

print("\nLOSO Child-level Classification Report:")
print(classification_report(
    loso_results_model3["y_true_child"],
    loso_results_model3["y_pred_child"],
    zero_division=0
))

print("LOSO Child-level Accuracy:", model3_loso_acc)
print("LOSO Child-level UAR:", model3_loso_uar)
print("LOSO Class 1 Recall:", model3_loso_recall1)

loso_results_dict["CatBoost + eGeMAPS"] = {
    "Accuracy": model3_loso_acc,
    "UAR": model3_loso_uar,
    "Recall_class1": model3_loso_recall1
}

In [ ]:
# preparing full DEV and final test
X_train_full = egemaps_dev[feature_cols].copy()
y_train_full = egemaps_dev["class_label"].values

X_test_final = egemaps_final_test[feature_cols].copy()
y_test_final = egemaps_final_test["class_label"].values
groups_test_final = egemaps_final_test["global_speaker_id"].values

# Preprocessing fitted on DEV only

lower = X_train_full.quantile(0.01)
upper = X_train_full.quantile(0.90)

X_train_clip = X_train_full.clip(lower=lower, upper=upper, axis=1)
X_test_clip = X_test_final.clip(lower=lower, upper=upper, axis=1)

corr_drop_cols = get_high_corr_cols(
    X_train_clip,
    threshold=0.95,
    protected_features=protected_features
)

X_train_corr = X_train_clip.drop(columns=corr_drop_cols)
X_test_corr = X_test_clip.drop(columns=corr_drop_cols)

In [ ]:
# Train CatBoost on full DEV

cat_final = CatBoostClassifier(
    iterations=500,
    learning_rate=0.03,
    depth=6,
    loss_function="Logloss",
    eval_metric="Recall",
    class_weights=[1, 2],
    random_seed=69,
    verbose=0
)

cat_final.fit(X_train_corr, y_train_full)

# Predict final test kids

y_pred_final_model3 = cat_final.predict(X_test_corr).astype(int).ravel()



In [ ]:
# Utterance-level results

model3_utt_acc = accuracy_score(y_test_final, y_pred_final_model3)

model3_utt_uar = recall_score(
    y_test_final,
    y_pred_final_model3,
    average="macro",
    zero_division=0
)

model3_utt_recall1 = recall_score(
    y_test_final,
    y_pred_final_model3,
    pos_label=1,
    average="binary",
    zero_division=0
)

print("Model 3: eGeMAPS + CatBoost Final Test Utterance-level")

print("\nUtterance-level Confusion Matrix:")
print(confusion_matrix(y_test_final, y_pred_final_model3))

print("\nUtterance-level Classification Report:")
print(classification_report(
    y_test_final,
    y_pred_final_model3,
    zero_division=0
))

print("Utterance-level Accuracy:", model3_utt_acc)
print("Utterance-level UAR:", model3_utt_uar)
print("Utterance-level Class 1 Recall:", model3_utt_recall1)

In [ ]:
# results breakdown

final_pred_df_model3 = pd.DataFrame({
    "global_speaker_id": groups_test_final,
    "y_true": y_test_final,
    "y_pred": y_pred_final_model3
})

child_results_model3 = []

for speaker_id, group in final_pred_df_model3.groupby("global_speaker_id"):

    child_results_model3.append({
        "global_speaker_id": speaker_id,
        "y_true_child": group["y_true"].mode()[0],
        "y_pred_child": group["y_pred"].mode()[0],
        "n_utterances": len(group),
        "utterance_accuracy": accuracy_score(group["y_true"], group["y_pred"])
    })

child_results_model3 = pd.DataFrame(child_results_model3)

child_results_model3

In [ ]:
# Evaluate and store child-level test result

model3_test_acc = accuracy_score(
    child_results_model3["y_true_child"],
    child_results_model3["y_pred_child"]
)

model3_test_uar = recall_score(
    child_results_model3["y_true_child"],
    child_results_model3["y_pred_child"],
    average="macro",
    zero_division=0
)

model3_test_recall1 = recall_score(
    child_results_model3["y_true_child"],
    child_results_model3["y_pred_child"],
    pos_label=1,
    average="binary",
    zero_division=0
)

print("Model 3: eGeMAPS + CatBoost Final Test Child-level")

print("\nFinal Test Child-level Confusion Matrix:")
print(confusion_matrix(
    child_results_model3["y_true_child"],
    child_results_model3["y_pred_child"]
))

print("\nFinal Test Child-level Classification Report:")
print(classification_report(
    child_results_model3["y_true_child"],
    child_results_model3["y_pred_child"],
    zero_division=0
))

print("Final Test Child-level Accuracy:", model3_test_acc)
print("Final Test Child-level UAR:", model3_test_uar)
print("Final Test Class 1 Recall:", model3_test_recall1)

test_results_dict["CatBoost + eGeMAPS"] = {
    "Accuracy": model3_test_acc,
    "UAR": model3_test_uar,
    "Recall_class1": model3_test_recall1,
    "n_kids": child_results_model3.shape[0]
}

#### Note: Using the ensemble based model helped improve the performance for SSD recall from 66% to 71% on Utterance level and 75% from 64% at Child level as compared to SVM with no SMOTE and with SMOTE.

### Model 4: Multi Layer Perceptron (MLP) on eGeMAPS features

In [ ]:
print("Model 4: eGeMAPS + MLP")

logo = LeaveOneGroupOut()
loso_results_model4 = []

for fold, (train_idx, test_idx) in enumerate(
    logo.split(X_dev_df, y_dev, groups_dev), start=1
):

    X_train = X_dev_df.iloc[train_idx].copy()
    X_test = X_dev_df.iloc[test_idx].copy()

    y_train = y_dev[train_idx]
    y_test = y_dev[test_idx]

    test_speaker = groups_dev[test_idx][0]

    # Outlier clipping fitted on training fold only
    lower = X_train.quantile(0.01)
    upper = X_train.quantile(0.90)

    X_train_clip = X_train.clip(lower=lower, upper=upper, axis=1)
    X_test_clip = X_test.clip(lower=lower, upper=upper, axis=1)

    # Correlation filtering fitted on training fold only
    corr_drop_cols = get_high_corr_cols(
        X_train_clip,
        threshold=0.95,
        protected_features=protected_features
    )

    X_train_corr = X_train_clip.drop(columns=corr_drop_cols)
    X_test_corr = X_test_clip.drop(columns=corr_drop_cols)

    # MLP needs scaling
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_corr)
    X_test_scaled = scaler.transform(X_test_corr)

    mlp_model = MLPClassifier(
        hidden_layer_sizes=(32, 16),
        activation="relu",
        solver="adam",
        alpha=0.01,
        learning_rate_init=0.001,
        max_iter=1000,
        early_stopping=True,
        validation_fraction=0.15,
        random_state=69
    )

    mlp_model.fit(X_train_scaled, y_train)

    y_pred = mlp_model.predict(X_test_scaled).astype(int).ravel()

    y_true_child = pd.Series(y_test).mode()[0]
    y_pred_child = pd.Series(y_pred).mode()[0]

    loso_results_model4.append({
        "fold": fold,
        "test_speaker": test_speaker,
        "y_true_child": y_true_child,
        "y_pred_child": y_pred_child,
        "n_utterances": len(y_test),
        "utterance_accuracy": accuracy_score(y_test, y_pred),
        "features_after_corr": X_train_corr.shape[1]
    })

loso_results_model4 = pd.DataFrame(loso_results_model4)

In [ ]:
# child level results on LOSO

model4_loso_acc = accuracy_score(
    loso_results_model4["y_true_child"],
    loso_results_model4["y_pred_child"]
)

model4_loso_uar = recall_score(
    loso_results_model4["y_true_child"],
    loso_results_model4["y_pred_child"],
    average="macro",
    zero_division=0
)

model4_loso_recall1 = recall_score(
    loso_results_model4["y_true_child"],
    loso_results_model4["y_pred_child"],
    pos_label=1,
    average="binary",
    zero_division=0
)

print("Model 4: eGeMAPS + MLP")

print("\nLOSO Child-level Confusion Matrix:")
print(confusion_matrix(
    loso_results_model4["y_true_child"],
    loso_results_model4["y_pred_child"]
))

print("\nLOSO Child-level Classification Report:")
print(classification_report(
    loso_results_model4["y_true_child"],
    loso_results_model4["y_pred_child"],
    zero_division=0
))

print("LOSO Child-level Accuracy:", model4_loso_acc)
print("LOSO Child-level UAR:", model4_loso_uar)
print("LOSO Class 1 Recall:", model4_loso_recall1)

loso_results_dict["MLP + eGeMAPS"] = {
    "Accuracy": model4_loso_acc,
    "UAR": model4_loso_uar,
    "Recall_class1": model4_loso_recall1
}

In [ ]:
# Prepare full DEV and final test

X_train_full = egemaps_dev[feature_cols].copy()
y_train_full = egemaps_dev["class_label"].values

X_test_final = egemaps_final_test[feature_cols].copy()
y_test_final = egemaps_final_test["class_label"].values
groups_test_final = egemaps_final_test["global_speaker_id"].values

# Preprocessing fitted on DEV only

lower = X_train_full.quantile(0.01)
upper = X_train_full.quantile(0.90)

X_train_clip = X_train_full.clip(lower=lower, upper=upper, axis=1)
X_test_clip = X_test_final.clip(lower=lower, upper=upper, axis=1)

corr_drop_cols = get_high_corr_cols(
    X_train_clip,
    threshold=0.95,
    protected_features=protected_features
)

X_train_corr = X_train_clip.drop(columns=corr_drop_cols)
X_test_corr = X_test_clip.drop(columns=corr_drop_cols)

# Scaling fitted on DEV only

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_corr)
X_test_scaled = scaler.transform(X_test_corr)

In [ ]:
# Train MLP on full DEV

mlp_final = MLPClassifier(
    hidden_layer_sizes=(32, 16),
    activation="relu",
    solver="adam",
    alpha=0.01,
    learning_rate_init=0.001,
    max_iter=1000,
    early_stopping=True,
    validation_fraction=0.15,
    random_state=69
)

mlp_final.fit(X_train_scaled, y_train_full)

# Predict final test kids

y_pred_final_model4 = mlp_final.predict(X_test_scaled).astype(int).ravel()

In [ ]:
# Utterance-level results

model4_utt_acc = accuracy_score(y_test_final, y_pred_final_model4)

model4_utt_uar = recall_score(
    y_test_final,
    y_pred_final_model4,
    average="macro",
    zero_division=0
)

model4_utt_recall1 = recall_score(
    y_test_final,
    y_pred_final_model4,
    pos_label=1,
    average="binary",
    zero_division=0
)

print("Model 4: eGeMAPS + MLP Final Test Utterance-level")

print("\nUtterance-level Confusion Matrix:")
print(confusion_matrix(y_test_final, y_pred_final_model4))

print("\nUtterance-level Classification Report:")
print(classification_report(
    y_test_final,
    y_pred_final_model4,
    zero_division=0
))

print("Utterance-level Accuracy:", model4_utt_acc)
print("Utterance-level UAR:", model4_utt_uar)
print("Utterance-level Class 1 Recall:", model4_utt_recall1)

In [ ]:
# results breakdown

final_pred_df_model4 = pd.DataFrame({
    "global_speaker_id": groups_test_final,
    "y_true": y_test_final,
    "y_pred": y_pred_final_model4
})

child_results_model4 = []

for speaker_id, group in final_pred_df_model4.groupby("global_speaker_id"):

    child_results_model4.append({
        "global_speaker_id": speaker_id,
        "y_true_child": group["y_true"].mode()[0],
        "y_pred_child": group["y_pred"].mode()[0],
        "n_utterances": len(group),
        "utterance_accuracy": accuracy_score(group["y_true"], group["y_pred"])
    })

child_results_model4 = pd.DataFrame(child_results_model4)

child_results_model4

In [ ]:
# Evaluate and store child-level test result

model4_test_acc = accuracy_score(
    child_results_model4["y_true_child"],
    child_results_model4["y_pred_child"]
)

model4_test_uar = recall_score(
    child_results_model4["y_true_child"],
    child_results_model4["y_pred_child"],
    average="macro",
    zero_division=0
)

model4_test_recall1 = recall_score(
    child_results_model4["y_true_child"],
    child_results_model4["y_pred_child"],
    pos_label=1,
    average="binary",
    zero_division=0
)

print("Model 4: eGeMAPS + MLP Final Test Child-level")

print("\nFinal Test Child-level Confusion Matrix:")
print(confusion_matrix(
    child_results_model4["y_true_child"],
    child_results_model4["y_pred_child"]
))

print("\nFinal Test Child-level Classification Report:")
print(classification_report(
    child_results_model4["y_true_child"],
    child_results_model4["y_pred_child"],
    zero_division=0
))

print("Final Test Child-level Accuracy:", model4_test_acc)
print("Final Test Child-level UAR:", model4_test_uar)
print("Final Test Class 1 Recall:", model4_test_recall1)

test_results_dict["MLP + eGeMAPS"] = {
    "Accuracy": model4_test_acc,
    "UAR": model4_test_uar,
    "Recall_class1": model4_test_recall1,
    "n_kids": child_results_model4.shape[0]
}

### Model 5: wav2vec 2.0 embeddings + SVM

#### The following code contains the helper function to extract the wav2vec 2.0 embeddings using the processed speech recordings from the master metadata table. It is already commented out as it takes alot of time to extract embeddings. If wanted to test, select command + / (On Mac) to remove the comments and then run.

In [ ]:
# # loading initial metadata
# df = pd.read_csv("master_metadata.csv")

# print("Initial dataframe shape:", df.shape)
# df.head()

# # Device setup
# device = "cuda" if torch.cuda.is_available() else "cpu"
# print("Using device:", device)

# # loading Wav2Vec2 model
# model_name = "facebook/wav2vec2-base-960h"
# processor = Wav2Vec2Processor.from_pretrained(model_name)
# model = Wav2Vec2Model.from_pretrained(model_name).to(device)
# model.eval()
# print("Wav2Vec2 model loaded successfully.")

# # Wav2Vec2 embedding extraction function
# def extract_wav2vec_embedding(file_path, processor, model, device):
#     try:
#         # loading audio and resample to 16 kHz
#         audio, sr = librosa.load(file_path, sr=16000)
#         # preparing input for Wav2Vec2
#         inputs = processor(
#             audio,
#             sampling_rate=16000,
#             return_tensors="pt")
#         input_values = inputs.input_values.to(device)
#         # extracting hidden states
#         with torch.no_grad():
#             outputs = model(input_values)
#         hidden_states = outputs.last_hidden_state
#         # mean pooling across time dimension
#         embedding = hidden_states.mean(dim=1).squeeze().cpu().numpy()
#         return embedding
    
#     except Exception as e:
#         print(f"Error processing {file_path}: {repr(e)}")
#         return None
    
# # extracting embeddings for full dataframe

# df["wav2vec_embedding"] = df["processed_child_wav_path"].apply(
#     lambda x: extract_wav2vec_embedding(x, processor, model, device))
# print("Embedding extraction completed.")

# # checking any failed extractions
# failed_count = df["wav2vec_embedding"].isnull().sum()
# successful_count = len(df) - failed_count
# print("Failed extractions:", failed_count)
# print("Successful extractions:", successful_count)

# # removing failed extractions if any
# df = df[df["wav2vec_embedding"].notnull()].copy()
# print("Final dataframe shape after removing failed rows:", df.shape)

# # storing the embeddings to csv
# df["wav2vec_embedding"] = df["wav2vec_embedding"].apply(lambda x: x.tolist())

# # selecting only required columns
# final_columns = [
#     "dataset_name",
#     "speaker_id",
#     "full_id",
#     "processed_child_wav_path",
#     "class_label",
#     "wav2vec_embedding"]

# df_embedding = df[final_columns].copy()
# df_embedding.head()

# # saving final embeddings CSV
# df_embedding.to_csv("df_embedding_extracted.csv", index=False)
# print("Final saved dataframe shape:", df_embedding.shape)

#### For wav2vec 2.0 embeddings, these were already extracted using the above helper function and saved as a seperate CSV file, so using the embeddings from the stored csv file to allow demonstration. If a user wanted to test the extraction function, then uncomment the above code and comment out the following code.

In [ ]:
df_embedding = pd.read_csv("df_embedding_extracted.csv")
df_embedding
df_embedding.isna().sum()

In [ ]:
# making a gloval speaker ID
df_embedding["global_speaker_id"] = df_embedding["dataset_name"] + "_" + df_embedding["speaker_id"].astype(str)
df_embedding

In [ ]:
# checking for null values
df_embedding.isna().sum()

In [ ]:
final_test_speakers

In [ ]:
# spliting the data (for validation and generalization testing)
df_embedding_test = df_embedding[df_embedding["global_speaker_id"].isin(final_test_speakers)].copy()
df_embedding_dev = df_embedding[~df_embedding["global_speaker_id"].isin(final_test_speakers)].copy()

In [ ]:
# helper function to convert back the embedding from string to numpy array if any
def convert_embedding(x):
    if isinstance(x, str):
        return np.array(ast.literal_eval(x), dtype=float)

    return np.array(x, dtype=float)


df_embedding_dev["wav2vec_embedding"] = df_embedding_dev["wav2vec_embedding"].apply(convert_embedding)
df_embedding_test["wav2vec_embedding"] = df_embedding_test["wav2vec_embedding"].apply(convert_embedding)

X2_dev = np.vstack(df_embedding_dev["wav2vec_embedding"].values)
y2_dev = df_embedding_dev["class_label"].values
groups2_dev = df_embedding_dev["global_speaker_id"].values

X2_test = np.vstack(df_embedding_test["wav2vec_embedding"].values)
y2_test = df_embedding_test["class_label"].values
groups2_test = df_embedding_test["global_speaker_id"].values

print("X2_dev shape:", X2_dev.shape)
print("X2_test shape:", X2_test.shape)

In [ ]:
print(X2_dev.shape)
print(X2_dev.dtype)

#### Model Training on SubSet

In [ ]:
logo = LeaveOneGroupOut()

loso_results_wav2vec = []
loso_utterance_preds_wav2vec = []

for fold, (train_idx, test_idx) in enumerate(
    logo.split(X2_dev, y2_dev, groups2_dev), start=1
):

    X_train, X_test = X2_dev[train_idx], X2_dev[test_idx]
    y_train, y_test = y2_dev[train_idx], y2_dev[test_idx]
    test_speaker = groups2_dev[test_idx][0]

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    svm = SVC(
        kernel="rbf",
        class_weight="balanced",
        random_state=69
    )

    svm.fit(X_train_scaled, y_train)

    y_pred = svm.predict(X_test_scaled)

    # store utterance-level predictions
    temp_utt = pd.DataFrame({
        "fold": fold,
        "test_speaker": test_speaker,
        "global_speaker_id": groups2_dev[test_idx],
        "y_true": y_test,
        "y_pred": y_pred
    })

    loso_utterance_preds_wav2vec.append(temp_utt)

    # child-level majority vote
    y_true_child = pd.Series(y_test).mode()[0]
    y_pred_child = pd.Series(y_pred).mode()[0]

    loso_results_wav2vec.append({
        "fold": fold,
        "test_speaker": test_speaker,
        "y_true_child": y_true_child,
        "y_pred_child": y_pred_child,
        "n_utterances": len(y_test),
        "utterance_accuracy": accuracy_score(y_test, y_pred)
    })

loso_results_wav2vec = pd.DataFrame(loso_results_wav2vec)
loso_utterance_preds_wav2vec = pd.concat(loso_utterance_preds_wav2vec, ignore_index=True)

In [ ]:
wav2vec_loso_utt_acc = accuracy_score(
    loso_utterance_preds_wav2vec["y_true"],
    loso_utterance_preds_wav2vec["y_pred"]
)

wav2vec_loso_utt_uar = recall_score(
    loso_utterance_preds_wav2vec["y_true"],
    loso_utterance_preds_wav2vec["y_pred"],
    average="macro",
    zero_division=0
)

wav2vec_loso_utt_recall1 = recall_score(
    loso_utterance_preds_wav2vec["y_true"],
    loso_utterance_preds_wav2vec["y_pred"],
    pos_label=1,
    average="binary",
    zero_division=0
)

print("Wav2Vec SVM LOSO Utterance-level")

print("\nUtterance-level Confusion Matrix:")
print(confusion_matrix(
    loso_utterance_preds_wav2vec["y_true"],
    loso_utterance_preds_wav2vec["y_pred"]
))

print("\nUtterance-level Classification Report:")
print(classification_report(
    loso_utterance_preds_wav2vec["y_true"],
    loso_utterance_preds_wav2vec["y_pred"],
    zero_division=0
))

print("Utterance-level Accuracy:", wav2vec_loso_utt_acc)
print("Utterance-level UAR:", wav2vec_loso_utt_uar)
print("Utterance-level Class 1 Recall:", wav2vec_loso_utt_recall1)

In [ ]:
wav2vec_loso_acc = accuracy_score(
    loso_results_wav2vec["y_true_child"],
    loso_results_wav2vec["y_pred_child"]
)

wav2vec_loso_uar = recall_score(
    loso_results_wav2vec["y_true_child"],
    loso_results_wav2vec["y_pred_child"],
    average="macro",
    zero_division=0
)

wav2vec_loso_recall1 = recall_score(
    loso_results_wav2vec["y_true_child"],
    loso_results_wav2vec["y_pred_child"],
    pos_label=1,
    average="binary",
    zero_division=0
)

print("Wav2Vec SVM LOSO Child-level")

print("\nChild-level Confusion Matrix:")
print(confusion_matrix(
    loso_results_wav2vec["y_true_child"],
    loso_results_wav2vec["y_pred_child"]
))

print("\nChild-level Classification Report:")
print(classification_report(
    loso_results_wav2vec["y_true_child"],
    loso_results_wav2vec["y_pred_child"],
    zero_division=0
))

print("Child-level Accuracy:", wav2vec_loso_acc)
print("Child-level UAR:", wav2vec_loso_uar)
print("Child-level Class 1 Recall:", wav2vec_loso_recall1)

loso_results_dict["wav2vec_svm"] = {
    "Accuracy": wav2vec_loso_acc,
    "UAR": wav2vec_loso_uar,
    "Recall_class1": wav2vec_loso_recall1
}

In [ ]:
# now training on whole dataset
scaler_final = StandardScaler()

X2_dev_scaled = scaler_final.fit_transform(X2_dev)
X2_test_scaled = scaler_final.transform(X2_test)

svm_final_wav2vec = SVC(
    kernel="rbf",
    class_weight="balanced",
    random_state=42
)

svm_final_wav2vec.fit(X2_dev_scaled, y2_dev)

y2_test_pred = svm_final_wav2vec.predict(X2_test_scaled)


In [ ]:
wav2vec_test_utt_acc = accuracy_score(y2_test, y2_test_pred)

wav2vec_test_utt_uar = recall_score(
    y2_test,
    y2_test_pred,
    average="macro",
    zero_division=0
)

wav2vec_test_utt_recall1 = recall_score(
    y2_test,
    y2_test_pred,
    pos_label=1,
    average="binary",
    zero_division=0
)

print("Wav2Vec SVM Final Test Utterance-level")

print("\nUtterance-level Confusion Matrix:")
print(confusion_matrix(y2_test, y2_test_pred))

print("\nUtterance-level Classification Report:")
print(classification_report(
    y2_test,
    y2_test_pred,
    zero_division=0
))

print("Utterance-level Accuracy:", wav2vec_test_utt_acc)
print("Utterance-level UAR:", wav2vec_test_utt_uar)
print("Utterance-level Class 1 Recall:", wav2vec_test_utt_recall1)

In [ ]:
# result breakdown
wav2vec_test_pred_df = pd.DataFrame({
    "global_speaker_id": groups2_test,
    "y_true": y2_test,
    "y_pred": y2_test_pred
})

child_results_wav2vec = []

for speaker_id, group in wav2vec_test_pred_df.groupby("global_speaker_id"):

    child_results_wav2vec.append({
        "global_speaker_id": speaker_id,
        "y_true_child": group["y_true"].mode()[0],
        "y_pred_child": group["y_pred"].mode()[0],
        "n_utterances": len(group),
        "utterance_accuracy": accuracy_score(group["y_true"], group["y_pred"])})

child_results_wav2vec = pd.DataFrame(child_results_wav2vec)

child_results_wav2vec

In [ ]:
wav2vec_test_acc = accuracy_score(
    child_results_wav2vec["y_true_child"],
    child_results_wav2vec["y_pred_child"]
)

wav2vec_test_uar = recall_score(
    child_results_wav2vec["y_true_child"],
    child_results_wav2vec["y_pred_child"],
    average="macro",
    zero_division=0
)

wav2vec_test_recall1 = recall_score(
    child_results_wav2vec["y_true_child"],
    child_results_wav2vec["y_pred_child"],
    pos_label=1,
    average="binary",
    zero_division=0
)

print("Wav2Vec SVM Final Test Child-level")

print("\nChild-level Confusion Matrix:")
print(confusion_matrix(
    child_results_wav2vec["y_true_child"],
    child_results_wav2vec["y_pred_child"]
))

print("\nChild-level Classification Report:")
print(classification_report(
    child_results_wav2vec["y_true_child"],
    child_results_wav2vec["y_pred_child"],
    zero_division=0
))

print("Child-level Accuracy:", wav2vec_test_acc)
print("Child-level UAR:", wav2vec_test_uar)
print("Child-level Class 1 Recall:", wav2vec_test_recall1)

test_results_dict["Support Vector Machine + wav2vec 2.0"] = {
    "Accuracy": wav2vec_test_acc,
    "UAR": wav2vec_test_uar,
    "Recall_class1": wav2vec_test_recall1,
    "n_kids": child_results_wav2vec.shape[0]
}

**Converting results to dictionary**

In [ ]:
df_loso_results = pd.DataFrame(loso_results_dict).T
print("LOSO Validation Results on Child Level")
df_loso_results

In [ ]:
df_kids_results = pd.DataFrame(test_results_dict).T
print("Held Out Results on Child Level")
df_kids_results


#### Note: The final model SVM with wav2vec 2.0 embeddings have given the best performance on validation level and on generalised test data, so it will act as a classifier in our pipeline.

In [ ]:
# storing the best model for streamlit
joblib.dump(svm_final_wav2vec , "wav2vec_classifier.joblib")
joblib.dump(scaler_final, "wav2vec_scaler.joblib")

## 7 - Feature Space Seperation (t-SNE Plot) between eGeMAPS & wav2vec 2.0 embeddings
- This is done to understand the the nature of results between two different feature set and why wav2vec 2.0 is generating better results compared to eGeMAPS features.
- Since it is not possible to visualize all the features at the recording level, so the features are visualize at child level, by taking the mean of the features values associated with the child speaker.
- Plotly library is used to draw the plot

In [ ]:
# eGeMAPS: preparing child level features
non_feature_cols = ["full_id", "global_speaker_id", "class_label"]

non_feature_cols = [
    c for c in non_feature_cols
    if c in egemaps_dev.columns]

feature_cols = [
    c for c in egemaps_dev.columns
    if c not in non_feature_cols]

# aggregating the mean of the eGeMAPS features associated with the child
child_egemaps = egemaps_dev.groupby("global_speaker_id").agg(
    class_label=("class_label", "first"),
    **{col: (col, "mean") for col in feature_cols}).reset_index()

X_ege = child_egemaps[feature_cols]
y_ege = child_egemaps["class_label"]

X_ege_scaled = StandardScaler().fit_transform(X_ege)

X_ege_tsne = TSNE(
    n_components=2, # for 2D plot, only reducing it to 2 components
    perplexity=10,
    learning_rate="auto",
    init="pca",
    random_state=69).fit_transform(X_ege_scaled)


# wav2vec 2.0 embeddings 
X_df = pd.DataFrame(X2_dev_scaled)

X_df["speaker_id"] = groups_dev
X_df["class_label"] = y_dev

# taking the mean of embeddings
child_df = X_df.groupby("speaker_id").mean().reset_index()

X_wav = child_df.drop(columns=["speaker_id", "class_label"])
y_wav = child_df["class_label"]

X_wav_scaled = StandardScaler().fit_transform(X_wav)

X_wav_tsne = TSNE(
    n_components=2,
    perplexity=10,
    learning_rate="auto",
    init="pca",
    random_state=42
).fit_transform(X_wav_scaled)

# preparing plotly dataframe

class_map = {
    0: "TD",
    1: "SSD"}

ege_plot = pd.DataFrame({
    "Dim 1": X_ege_tsne[:, 0],
    "Dim 2": X_ege_tsne[:, 1],
    "Class": y_ege.map(class_map),
    "Speaker ID": child_egemaps["global_speaker_id"]})

wav_plot = pd.DataFrame({
    "Dim 1": X_wav_tsne[:, 0],
    "Dim 2": X_wav_tsne[:, 1],
    "Class": y_wav.map(class_map),
    "Speaker ID": child_df["speaker_id"]})

# plot styling and layout
colors = {"TD": "#4C78A8", "SSD": "#E45756"}

fig = make_subplots(rows=1,cols=2,subplot_titles=("eGeMAPS Features","wav2vec2 Embeddings"),horizontal_spacing=0.08)

# scatter plot
for col_idx, data in enumerate([ege_plot, wav_plot], start=1):

    for cls in ["TD", "SSD"]:
        subset = data[data["Class"] == cls]
        fig.add_trace(
            go.Scatter(
                x=subset["Dim 1"],
                y=subset["Dim 2"],
                mode="markers",
                name=cls if col_idx == 1 else None,
                showlegend=True if col_idx == 1 else False,
                marker=dict(
                    size=12,
                    color=colors[cls],
                    opacity=0.85,
                    line=dict(
                        width=1.2,
                        color="white"
                    )
                ),

                customdata=subset[["Speaker ID", "Class"]],
                hovertemplate=(
                    "<b>Speaker ID:</b> %{customdata[0]}<br>"
                    "<b>Class:</b> %{customdata[1]}<br>"
                    "<b>Dimension 1:</b> %{x:.2f}<br>"
                    "<b>Dimension 2:</b> %{y:.2f}"
                    "<extra></extra>"
                )
            ),
            row=1,
            col=col_idx
        )

# final layout
fig.update_layout(
    title=dict(text="Child-Level t-SNE Visualisation", x=0.5, y=0.98, font=dict(size=28, family="Arial Black")),
    template="plotly_white",
    width=1200,
    height=560,
    legend=dict(title="Class Label", orientation="h", x=0.5, y=1.1, xanchor="center", yanchor="bottom"),
    margin=dict(l=60, r=50, t=125, b=55))

fig.update_xaxes(title_text="Dimension 1",showgrid=True,zeroline=False)
fig.update_yaxes(title_text="Dimension 2",showgrid=True,zeroline=False)

fig.show()

#### The results does confirm that wav2vec 2.0 embeddings have a clear seperation between two classes compared to eGeMAP features, where a large overalap is happening. This indicate that deep embeddings are able to capture fine grain speech representation in the dataset, which hold better prediction strength for the classification of the Speech Sound Disorder, compared to eGeMAP features.

## 8- Speech Profile
- For Speech profile, eGeMAPS features are used, since they are explanable in nature as compared to wav2vec embeddings.
- Speech Profile is just to provide objective summary to the therapist and it cannot be used for the diagnosis or purely for classification purposes.

In [ ]:
profile_train_df = egemaps_dev.copy()
profile_test_df = egemaps_final_test.copy()

non_feature_cols = ["full_id", "global_speaker_id", "class_label"]
non_feature_cols = [c for c in non_feature_cols if c in profile_train_df.columns]

feature_cols = [
    c for c in profile_train_df.columns
    if c not in non_feature_cols]

print("Train/dev shape:", profile_train_df.shape)
print("Test shape:", profile_test_df.shape)
print("Number of features:", len(feature_cols))

#### Following 6 major and sub dimensions are defined based on the basic acoustic literature. The grouping is also performed based on the similarity of data types between related features in a certain dimension (mean, percentile, slopes, standard deviation).


In [ ]:
profile_dimensions = {
    "Pitch / Prosody": {
        "Pitch Level": [
            "F0semitoneFrom27.5Hz_sma3nz_amean",
            "F0semitoneFrom27.5Hz_sma3nz_percentile20.0",
            "F0semitoneFrom27.5Hz_sma3nz_percentile50.0",
            "F0semitoneFrom27.5Hz_sma3nz_percentile80.0",
        ],
        "Pitch Variability": [
            "F0semitoneFrom27.5Hz_sma3nz_stddevNorm",
            "F0semitoneFrom27.5Hz_sma3nz_pctlrange0-2",
        ],
        "Pitch Movement": [
            "F0semitoneFrom27.5Hz_sma3nz_meanRisingSlope",
            "F0semitoneFrom27.5Hz_sma3nz_stddevRisingSlope",
            "F0semitoneFrom27.5Hz_sma3nz_meanFallingSlope",
            "F0semitoneFrom27.5Hz_sma3nz_stddevFallingSlope",
        ],
    },

    "Loudness / Energy": {
        "Loudness Level": [
            "loudness_sma3_amean",
            "loudness_sma3_percentile20.0",
            "loudness_sma3_percentile50.0",
            "loudness_sma3_percentile80.0",
            "equivalentSoundLevel_dBp",
        ],
        "Loudness Variability": [
            "loudness_sma3_stddevNorm",
            "loudness_sma3_pctlrange0-2",
        ],
        "Loudness Dynamics": [
            "loudness_sma3_meanRisingSlope",
            "loudness_sma3_stddevRisingSlope",
            "loudness_sma3_meanFallingSlope",
            "loudness_sma3_stddevFallingSlope",
        ],
        "Peak Rate": [
            "loudnessPeaksPerSec",
        ],
    },

    "Voice Quality": {
        "Vocal Stability": [
            "jitterLocal_sma3nz_amean",
            "jitterLocal_sma3nz_stddevNorm",
            "shimmerLocaldB_sma3nz_amean",
            "shimmerLocaldB_sma3nz_stddevNorm",
        ],
        "Harmonicity / Noise": [
            "HNRdBACF_sma3nz_amean",
            "HNRdBACF_sma3nz_stddevNorm",
        ],
        "Spectral Voice Quality": [
            "logRelF0-H1-H2_sma3nz_amean",
            "logRelF0-H1-H2_sma3nz_stddevNorm",
            "logRelF0-H1-A3_sma3nz_amean",
            "logRelF0-H1-A3_sma3nz_stddevNorm",
        ],
    },

    "Articulation / Vowel Quality": {
        "Formant Frequency": [
            "F1frequency_sma3nz_amean",
            "F1frequency_sma3nz_stddevNorm",
            "F2frequency_sma3nz_amean",
            "F2frequency_sma3nz_stddevNorm",
            "F3frequency_sma3nz_amean",
            "F3frequency_sma3nz_stddevNorm",
        ],
        "Formant Bandwidth": [
            "F1bandwidth_sma3nz_amean",
            "F1bandwidth_sma3nz_stddevNorm",
            "F2bandwidth_sma3nz_amean",
            "F2bandwidth_sma3nz_stddevNorm",
            "F3bandwidth_sma3nz_amean",
            "F3bandwidth_sma3nz_stddevNorm",
        ],
        "Formant Amplitude": [
            "F1amplitudeLogRelF0_sma3nz_amean",
            "F1amplitudeLogRelF0_sma3nz_stddevNorm",
            "F2amplitudeLogRelF0_sma3nz_amean",
            "F2amplitudeLogRelF0_sma3nz_stddevNorm",
            "F3amplitudeLogRelF0_sma3nz_amean",
            "F3amplitudeLogRelF0_sma3nz_stddevNorm",
        ],
    },

    "Spectral / Phonation": {
        "Spectral Balance": [
            "alphaRatioV_sma3nz_amean",
            "alphaRatioV_sma3nz_stddevNorm",
            "alphaRatioUV_sma3nz_amean",
            "hammarbergIndexV_sma3nz_amean",
            "hammarbergIndexV_sma3nz_stddevNorm",
            "hammarbergIndexUV_sma3nz_amean",
        ],
        "Spectral Slope": [
            "slopeV0-500_sma3nz_amean",
            "slopeV0-500_sma3nz_stddevNorm",
            "slopeV500-1500_sma3nz_amean",
            "slopeV500-1500_sma3nz_stddevNorm",
            "slopeUV0-500_sma3nz_amean",
            "slopeUV500-1500_sma3nz_amean",
        ],
        "Spectral Flux": [
            "spectralFlux_sma3_amean",
            "spectralFlux_sma3_stddevNorm",
            "spectralFluxV_sma3nz_amean",
            "spectralFluxV_sma3nz_stddevNorm",
            "spectralFluxUV_sma3nz_amean",
        ],
        "MFCC Shape": [
            "mfcc1_sma3_amean",
            "mfcc1_sma3_stddevNorm",
            "mfcc2_sma3_amean",
            "mfcc2_sma3_stddevNorm",
            "mfcc3_sma3_amean",
            "mfcc3_sma3_stddevNorm",
            "mfcc4_sma3_amean",
            "mfcc4_sma3_stddevNorm",
            "mfcc1V_sma3nz_amean",
            "mfcc1V_sma3nz_stddevNorm",
            "mfcc2V_sma3nz_amean",
            "mfcc2V_sma3nz_stddevNorm",
            "mfcc3V_sma3nz_amean",
            "mfcc3V_sma3nz_stddevNorm",
            "mfcc4V_sma3nz_amean",
            "mfcc4V_sma3nz_stddevNorm",
        ],
    },

    "Fluency / Rhythm": {
        "Voicing Rate": [
            "VoicedSegmentsPerSec",
        ],
        "Voiced Duration": [
            "MeanVoicedSegmentLengthSec",
            "StddevVoicedSegmentLengthSec",
        ],
        "Unvoiced Duration": [
            "MeanUnvoicedSegmentLength",
            "StddevUnvoicedSegmentLength",
        ],
        "Peak Rhythm": [
            "loudnessPeaksPerSec",
        ],
    },
}

In [ ]:
clean_dimensions = {}

for dimension, subgroups in profile_dimensions.items():
    clean_dimensions[dimension] = {}

    for subgroup, features in subgroups.items():
        available_features = [f for f in features if f in feature_cols]

        if len(available_features) > 0:
            clean_dimensions[dimension][subgroup] = available_features

clean_dimensions

In [ ]:
# seperating the children (development and test kid)
train_child_features = (
    profile_train_df
    .groupby("global_speaker_id")[feature_cols]
    .mean()
)

train_child_labels = (
    profile_train_df
    .groupby("global_speaker_id")["class_label"]
    .first()
)

test_child_features = (
    profile_test_df
    .groupby("global_speaker_id")[feature_cols]
    .mean()
)

print("Train/dev children:", train_child_features.shape[0])
print("Test children:", test_child_features.shape[0])

In [ ]:
# using the baseline children who have the normal speech (TD:0)
td_child_features = train_child_features[train_child_labels == 0]
td_mean = td_child_features.mean()
td_std = td_child_features.std().replace(0, 1)

print("TD reference children:", td_child_features.shape[0])

In [ ]:
# standardizing the data first
td_z_abs = ((td_child_features - td_mean) / td_std).abs()
test_z_abs = ((test_child_features - td_mean) / td_std).abs()

In [ ]:
# function to calculate the dimension score
def calculate_profile_scores(z_abs_df, dimensions_dict):
    profile_results = {}

    for dimension, subgroups in dimensions_dict.items():
        subgroup_scores = {}

        for subgroup, features in subgroups.items():
            subgroup_scores[subgroup] = z_abs_df[features].mean(axis=1)

        profile_results[dimension] = pd.DataFrame(subgroup_scores)

    return profile_results

td_profiles = calculate_profile_scores(td_z_abs, clean_dimensions)
test_profiles = calculate_profile_scores(test_z_abs, clean_dimensions)

# average TD reference profile for each dimension
td_baselines = {
    dimension: profile_df.mean()
    for dimension, profile_df in td_profiles.items()
}

In [ ]:
# for Plotly Speech Profile

pio.renderers.default = "notebook"
def plot_all_dimension_radars_plotly(child_id, td_baselines, test_profiles):

    dimensions = list(td_baselines.keys())
    n_dims = len(dimensions)

    n_cols = 3
    n_rows = int(np.ceil(n_dims / n_cols))

    specs = [[{"type": "polar"} for _ in range(n_cols)] for _ in range(n_rows)]

    # Bold subplot titles
    subplot_titles = [f"<b>{d}</b>" for d in dimensions]

    fig = make_subplots(
        rows=n_rows,
        cols=n_cols,
        specs=specs,
        subplot_titles=subplot_titles,
        horizontal_spacing=0.10,
        vertical_spacing=0.28
    )

    # Style subplot titles
    for annotation in fig.layout.annotations:
        annotation.font = dict(size=15)
        annotation.y = annotation.y + 0.04

    td_color = "#1f77b4"
    child_color = "#ff7f0e"

    for i, dimension in enumerate(dimensions):

        row = i // n_cols + 1
        col = i % n_cols + 1

        td_baseline = td_baselines[dimension]
        test_profile_df = test_profiles[dimension]

        labels = td_baseline.index.tolist()

        td_values = td_baseline.values.tolist()
        child_values = test_profile_df.loc[child_id, labels].values.tolist()

        # close radar shape
        labels_closed = labels + [labels[0]]

        td_closed = td_values + [td_values[0]]
        child_closed = child_values + [child_values[0]]

        fig.add_trace(
            go.Scatterpolar(
                r=td_closed,
                theta=labels_closed,
                mode="lines+markers",
                name="Average TD Reference",
                legendgroup="TD",
                showlegend=(i == 0),
                line=dict(color=td_color, width=2.5),
                marker=dict(color=td_color, size=6),
                fill="toself",
                fillcolor="rgba(31,119,180,0.12)"
            ),
            row=row,
            col=col
        )

        fig.add_trace(
            go.Scatterpolar(
                r=child_closed,
                theta=labels_closed,
                mode="lines+markers",
                name=f"Child: {child_id}",
                legendgroup="Child",
                showlegend=(i == 0),
                line=dict(color=child_color, width=2.5),
                marker=dict(color=child_color, size=6),
                fill="toself",
                fillcolor="rgba(255,127,14,0.18)"
            ),
            row=row,
            col=col
        )

    # shared radial scale
    all_values = []

    for dimension in dimensions:
        all_values.extend(td_baselines[dimension].values.tolist())
        all_values.extend(
            test_profiles[dimension].loc[child_id].values.tolist()
        )

    max_r = max(2.5, np.nanmax(all_values) + 0.3)

    polar_layout = dict(
        radialaxis=dict(
            range=[0, max_r],
            tickfont=dict(size=10),
            gridcolor="lightgray",
            linecolor="lightgray"
        ),
        angularaxis=dict(
            tickfont=dict(size=10),
            gridcolor="lightgray",
            linecolor="lightgray"
        ),
        bgcolor="rgba(240,245,252,0.85)"
    )

    for i in range(1, n_dims + 1):

        polar_name = "polar" if i == 1 else f"polar{i}"

        fig.update_layout(
            **{
                polar_name: polar_layout
            }
        )

    fig.update_layout(

        title=dict(
            text=f"<b>Speech Profile Radar Charts</b><br>TD Reference vs Test Child: {child_id}",
            x=0.5,
            y=0.98,
            xanchor="center",
            font=dict(size=20)
        ),

        height=1050,
        width=1400,

        legend=dict(
            orientation="h",
            x=0.5,
            y=1.10,
            xanchor="center",
            yanchor="bottom",
            font=dict(size=12)
        ),

        margin=dict(
            t=190,
            b=110,
            l=60,
            r=60
        )
    )

    fig.add_annotation(
        text="<b>Scores = mean absolute z-score from TD feature baseline | closer to centre = TD-like | further outward = greater deviation</b>",
        x=0.5,
        y=-0.08,
        xref="paper",
        yref="paper",
        showarrow=False,
        font=dict(size=13)
    )

    return fig

#### The following code is used to plot the final speech profile. It can be tested against any child speaker from the dataset by changing the child_id. The child_id must follow the following sequence:

- datasetname_childid

In [ ]:
plot_all_dimension_radars_plotly(
    child_id="upx_02F", # change the speaker ID to draw the profile for the tested kid
    td_baselines=td_baselines,
    test_profiles=test_profiles)

In [ ]:
final_test_speakers

In [ ]:
#  storing objects into .joblib files to reuse in Streamlit app
joblib.dump(td_mean, "td_mean_egemaps.joblib")
joblib.dump(td_std, "td_std_egemaps.joblib")
joblib.dump(td_baselines, "td_baselines.joblib")
joblib.dump(clean_dimensions, "clean_dimensions.joblib")